# 🌊 Bellinge — Full ML Pipeline

**Liquid Neural Networks for Urban Drainage Overflow Prediction**

This notebook runs the complete pipeline:
1. Environment setup & data loading from Google Drive
2. Single LNN model training
3. 5-member ensemble training
4. Overflow threshold selection on validation split
5. Model evaluation (depth + overflow metrics)
6. Ensemble evaluation
7. Robustness analysis (noise, missing data, FGSM)
8. Integrated Gradients feature attribution
9. Uncertainty estimation (MC Dropout + Delta method)
10. Save results back to Google Drive

---

**Prerequisites:**
- Upload `bellinge_final_regression.tar.gz` to `My Drive/bellinge/`
- Generate the archive locally with: `bash notebooks/prepare_colab_data.sh`
- Select a **GPU runtime** (Runtime → Change runtime type → T4 GPU)

---
## 0. Environment Setup

In [ ]:
# ── 0.1 Check GPU availability ──
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU detected: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("No GPU detected! Training will be very slow.")
    print("Go to: Runtime → Change runtime type → T4 GPU")

In [ ]:
import os

# ── 0.2 Mount Google Drive ──
from google.colab import drive

drive.mount("/content/drive")

# Verify data archive exists
DRIVE_DATA_DIR = "/content/drive/MyDrive/bellinge"
ARCHIVE_NAME = "bellinge_final_regression.tar.gz"
ARCHIVE_PATH = os.path.join(DRIVE_DATA_DIR, ARCHIVE_NAME)

if os.path.exists(ARCHIVE_PATH):
    size_mb = os.path.getsize(ARCHIVE_PATH) / 1e6
    print(f"Found data archive: {ARCHIVE_PATH} ({size_mb:.0f} MB)")
else:
    print(f"Archive not found at: {ARCHIVE_PATH}")
    print(f"Upload {ARCHIVE_NAME} to Google Drive: My Drive/bellinge/")

In [ ]:
# ── 0.3 Clone repository & install dependencies ──
import subprocess

REPO_URL = (
    "https://github.com/BuczynskiRafal/bellinge.git"  # ← UPDATE with your repo URL
)
REPO_DIR = "/content/bellinge"

if not os.path.exists(REPO_DIR):
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print(f"Repository already exists at {REPO_DIR}, pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ── 0.4 Install Python dependencies ──
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q captum>=0.7.0 scikit-learn>=1.2.0 imbalanced-learn>=0.10.0 \
    matplotlib>=3.6.0 seaborn>=0.12.0 h5py>=3.7.0 pyyaml>=6.0 scipy>=1.9.0

# Verify critical imports
import numpy as np
import torch
import yaml

print(f"torch={torch.__version__}, numpy={np.__version__}")
print(f"   CUDA available: {torch.cuda.is_available()}")

---
## 1. Data Download & Verification

In [ ]:
# ── 1.1 Unpack data archive ──
import tarfile

DATA_DIR = os.path.join(REPO_DIR, "data", "final_regression")
os.makedirs(DATA_DIR, exist_ok=True)

# Check if data is already unpacked
expected_file = os.path.join(DATA_DIR, "train_X_reduced.npy")
if os.path.exists(expected_file):
    print("Data already unpacked.")
else:
    print(f"Unpacking {ARCHIVE_PATH}...")
    with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
        tar.extractall(path=REPO_DIR)
    print("Data unpacked successfully.")

# Show contents
for f in sorted(os.listdir(DATA_DIR)):
    size_mb = os.path.getsize(os.path.join(DATA_DIR, f)) / 1e6
    print(f"  {f:40s} {size_mb:>8.1f} MB")

In [ ]:
# ── 1.2 Verify data contract ──
from src.data.load_regression_data import load_regression_data

for split in ["train", "val", "test"]:
    data = load_regression_data(split, use_reduced=True)
    X = data["X"]
    y_d = data["y_depths"]
    y_o = data["y_overflow"]
    fm = data["flood_mask"]
    overflow_pct = 100.0 * y_o.mean()
    print(
        f"{split:>5s}: X={X.shape}, y_depths={y_d.shape}, "
        f"y_overflow={y_o.shape}, flood_mask={fm.shape} "
        f"| overflow={overflow_pct:.1f}%"
    )

print("\nAll splits loaded and validated.")

---
## 2. Single LNN Model Training

Trains a single Liquid Neural Network using the canonical configuration.

**Architecture:** fast=128, slow=128, hidden=320, tau_mode=stepwise  
**Training:** Adam, cosine annealing, 200 epochs, early stopping (patience=20)

In [ ]:
# ── 2.1 Load config and prepare ──
from pathlib import Path

from experiments.regression_pipeline import (
    count_parameters,
    create_model,
    get_runtime_device,
    load_model_config,
    train_model,
)

MODEL_NAME = "lnn"
config = load_model_config(MODEL_NAME)
device = get_runtime_device(config)

print(f"Device: {device}")
print("Config:")
print(yaml.dump(config, default_flow_style=False))

# Preview model size
model_preview = create_model(MODEL_NAME, config)
n_params = count_parameters(model_preview)
print(f"\nModel parameters: {n_params:,}")
del model_preview

In [ ]:
# ── 2.2 Train single model ──
checkpoint_path = Path(config["output"]["checkpoint_dir"]) / "best_model.pt"

model, training_summary = train_model(
    MODEL_NAME,
    config,
    device=device,
    checkpoint_path=checkpoint_path,
)

print(f"\n{'='*50}")
print("TRAINING COMPLETE")
print(f"{'='*50}")
print(f"Best epoch:       {training_summary['best_epoch']}")
print(f"Best val loss:    {training_summary['best_val_loss']:.6f}")
print(f"Best val accuracy: {training_summary['best_val_accuracy']:.1f}%")
print(f"Epochs ran:       {training_summary['epochs_ran']}")
print(f"Parameters:       {training_summary['num_parameters']:,}")
print(f"Checkpoint:       {checkpoint_path}")

---
## 3. Ensemble Training (5 Members)

Trains 5 independent LNN models with seeds 42–46.  
Each member gets its own checkpoint under `artifacts/checkpoints/ensemble/lnn/seed_<N>/`.

In [ ]:
# ── 3.1 Train 5-member ensemble ──
from experiments.regression_pipeline import train_lnn_ensemble

ENSEMBLE_SEEDS = [42, 43, 44, 45, 46]

ensemble_payload = train_lnn_ensemble(
    seeds=ENSEMBLE_SEEDS,
    checkpoint_root="artifacts/checkpoints/ensemble",
)

print(f"\n{'='*60}")
print(f"ENSEMBLE TRAINING COMPLETE — {ensemble_payload['ensemble_size']} members")
print(f"{'='*60}")
for member in ensemble_payload["members"]:
    print(
        f"  seed={member['seed']:2d} | "
        f"best_epoch={member['best_epoch']:3d} | "
        f"val_loss={member['best_val_loss']:.6f} | "
        f"val_acc={member['best_val_accuracy']:.1f}%"
    )

---
## 4. Threshold Selection (Validation Split)

Finds the optimal overflow classification threshold by maximizing F1-score on the validation set.

In [ ]:
# ── 4.1 Select threshold on validation split ──
from experiments.regression_pipeline import select_overflow_threshold

threshold_payload, threshold_path = select_overflow_threshold(
    MODEL_NAME,
    split="val",
)

print(f"Optimal threshold: {threshold_payload['threshold']:.6f}")
print("Validation metrics at this threshold:")
for metric, value in threshold_payload["metrics"].items():
    print(f"  {metric:12s}: {value:.4f}")
print(f"\nSaved to: {threshold_path}")

---
## 5. Single Model Evaluation

Evaluates the trained LNN on the test split with depth regression, hydrological, and overflow metrics.

In [ ]:
# ── 5.1 Evaluate single model ──
import json

from experiments.regression_pipeline import (
    build_evaluation_payload,
    save_evaluation_payload,
)

eval_payload = build_evaluation_payload(MODEL_NAME, split="test")
metrics = eval_payload["metrics"]

print(f"{'='*50}")
print("DEPTH REGRESSION METRICS (Aggregated)")
print(f"{'='*50}")
for metric, value in metrics["depth_metrics"]["aggregated"].items():
    print(f"  {metric:12s}: {value:.4f}")

print(f"\n{'='*50}")
print("HYDROLOGICAL METRICS (Aggregated)")
print(f"{'='*50}")
for metric, value in metrics["hydrological_metrics"]["aggregated"].items():
    print(f"  {metric:25s}: {value:.4f}")

print(f"\n{'='*50}")
print("OVERFLOW CLASSIFICATION")
print(f"{'='*50}")
for metric, value in metrics["overflow_metrics"].items():
    print(f"  {metric:12s}: {value:.4f}")

print(
    f"\nThreshold: {metrics['overflow_threshold']:.6f} (source: {metrics['overflow_threshold_source']})"
)

# Save results
metrics_path, manifest_path = save_evaluation_payload(MODEL_NAME, eval_payload)
print(f"\nMetrics saved to:  {metrics_path}")
print(f"Manifest saved to: {manifest_path}")

---
## 6. Ensemble Evaluation

Loads all 5 ensemble members, aggregates predictions, selects threshold from averaged val probabilities.

In [ ]:
# ── 6.1 Evaluate LNN ensemble ──
from experiments.regression_pipeline import evaluate_lnn_ensemble

ensemble_eval_payload, ensemble_results_path = evaluate_lnn_ensemble(
    seeds=ENSEMBLE_SEEDS,
    checkpoint_root="artifacts/checkpoints/ensemble",
)

e_metrics = ensemble_eval_payload["ensemble_metrics"]

print(f"{'='*60}")
print(f"ENSEMBLE EVALUATION — {ensemble_eval_payload['ensemble_size']} members")
print(f"{'='*60}")
print(f"Overflow threshold: {ensemble_eval_payload['overflow_threshold']:.6f}")
print(f"(source: {ensemble_eval_payload['overflow_threshold_source']})")

print("\nDepth metrics (aggregated):")
for metric, value in e_metrics["depth_metrics"]["aggregated"].items():
    print(f"  {metric:12s}: {value:.4f}")

print("\nOverflow metrics:")
for metric, value in e_metrics["overflow_metrics"].items():
    print(f"  {metric:12s}: {value:.4f}")

print("\nPer-member performance:")
for member in ensemble_eval_payload["member_metrics"]:
    m = member["metrics"]
    nse = m["depth_metrics"]["aggregated"]["NSE"]
    f1 = m["overflow_metrics"]["F1"]
    print(f"  seed={member['seed']:2d} | NSE={nse:.4f} | F1={f1:.4f}")

print(f"\nResults saved to: {ensemble_results_path}")

---
## 7. Robustness Analysis

Tests model degradation under:
- **Gaussian noise** at SNR levels: 40, 30, 20, 10 dB
- **Missing data** (Bernoulli masking): 5%, 10%, 15%, 20%
- **FGSM** adversarial perturbations: ε = 0.001, 0.005, 0.01

In [ ]:
# ── 7.1 Run robustness evaluation ──
import subprocess

result = subprocess.run(
    [
        "python",
        "-m",
        "experiments.eval_robustness",
        "--model",
        "lnn",
        "--noise-db",
        "40",
        "30",
        "20",
        "10",
        "--missing-rates",
        "0.05",
        "0.10",
        "0.15",
        "0.20",
        "--fgsm-eps",
        "0.001",
        "0.005",
        "0.01",
    ],
    capture_output=True,
    text=True,
)

if result.returncode == 0:
    print("Robustness evaluation complete.")
else:
    print(f"Error:\n{result.stderr}")
    raise RuntimeError(result.stderr)

In [ ]:
# ── 7.2 Display robustness results ──
from pathlib import Path

robustness_file = Path("artifacts/results/robustness/lnn_robustness.json")
with open(robustness_file) as f:
    robustness = json.load(f)

baseline_nse = robustness["baseline_nse"]
print(f"Baseline NSE: {baseline_nse:.4f}\n")

print(f"{'─'*55}")
print(f"{'Gaussian Noise':^55}")
print(f"{'─'*55}")
print(f"{'SNR (dB)':>10}  {'NSE':>8}  {'ΔNSE':>8}  {'Degradation':>12}")
for r in robustness["gaussian_noise"]:
    pct = 100 * r["nse_drop"] / baseline_nse if baseline_nse != 0 else 0
    print(
        f"{r['snr_db']:>10.0f}  {r['nse']:>8.4f}  {r['nse_drop']:>8.4f}  {pct:>11.1f}%"
    )

print(f"\n{'─'*55}")
print(f"{'Missing Data (Bernoulli Masking)':^55}")
print(f"{'─'*55}")
print(f"{'Mask Rate':>10}  {'NSE':>8}  {'ΔNSE':>8}  {'Degradation':>12}")
for r in robustness["missing_data"]:
    pct = 100 * r["nse_drop"] / baseline_nse if baseline_nse != 0 else 0
    print(
        f"{r['mask_rate']:>10.0%}  {r['nse']:>8.4f}  {r['nse_drop']:>8.4f}  {pct:>11.1f}%"
    )

print(f"\n{'─'*55}")
print(f"{'FGSM Adversarial':^55}")
print(f"{'─'*55}")
print(f"{'Epsilon':>10}  {'NSE':>8}  {'ΔNSE':>8}  {'Degradation':>12}")
for r in robustness["fgsm"]:
    pct = 100 * r["nse_drop"] / baseline_nse if baseline_nse != 0 else 0
    print(
        f"{r['epsilon']:>10.3f}  {r['nse']:>8.4f}  {r['nse_drop']:>8.4f}  {pct:>11.1f}%"
    )

---
## 8. Integrated Gradients

Computes feature-level and timestep-level attributions for the overflow prediction head using Captum.

In [ ]:
# ── 8.1 Run IG analysis ──
result = subprocess.run(
    [
        "python",
        "-m",
        "experiments.eval_ig",
        "--model",
        "lnn",
        "--max-samples",
        "128",
    ],
    capture_output=True,
    text=True,
)

if result.returncode == 0:
    print("Integrated Gradients evaluation complete.")
else:
    print(f"Error:\n{result.stderr}")
    raise RuntimeError(result.stderr)

In [ ]:
# ── 8.2 Display feature importance ranking ──
import matplotlib.pyplot as plt

ig_file = Path("artifacts/results/ig/lnn_integrated_gradients.json")
with open(ig_file) as f:
    ig_results = json.load(f)

print(f"Samples: {ig_results['num_samples']}")
print(f"Mean convergence delta: {ig_results['mean_convergence_delta']:.6f}")

# Sort features by importance
features = ig_results["feature_importance"]
sorted_features = sorted(features.items(), key=lambda x: x[1], reverse=True)

print("\nTop 15 features by |attribution|:")
print(f"{'Feature':>35s}  {'Score':>10s}")
print(f"{'─'*50}")
for name, score in sorted_features[:15]:
    bar = "█" * int(score / sorted_features[0][1] * 30)
    print(f"{name:>35s}  {score:>10.6f}  {bar}")

# Bar chart
fig, ax = plt.subplots(figsize=(10, 8))
top_n = sorted_features[:20]
names = [n for n, _ in reversed(top_n)]
scores = [s for _, s in reversed(top_n)]
ax.barh(names, scores, color="#2196F3")
ax.set_xlabel("Mean |Attribution|")
ax.set_title("Integrated Gradients — Top 20 Features (Overflow Head)")
plt.tight_layout()
plt.savefig("artifacts/results/ig/feature_importance_chart.png", dpi=150)
plt.show()

---
## 9. Uncertainty Estimation

Two methods:
- **MC Dropout** — 100 stochastic forward passes
- **Delta Method** — first-order variance propagation through the Jacobian

In [ ]:
# ── 9.1 Run uncertainty analysis ──
from experiments.evaluate_lnn_uncertainty import main as run_uncertainty

run_uncertainty()
print("\nUncertainty evaluation complete.")

In [ ]:
# ── 9.2 Display uncertainty results ──
import glob

uncertainty_files = sorted(
    glob.glob("artifacts/results/uncertainty/lnn_uncertainty_metrics_*.json")
)
if not uncertainty_files:
    print("No uncertainty results found.")
else:
    latest = uncertainty_files[-1]
    with open(latest) as f:
        unc = json.load(f)

    print(f"Architecture: {unc['architecture']}")
    print(f"Test samples: {unc['test_samples']}")
    print(f"Confidence level: {unc['confidence_level']}")

    for method_name in ["mc_dropout", "delta_method"]:
        method = unc[method_name]
        print(f"\n{'='*50}")
        print(f"{method_name.upper().replace('_', ' ')}")
        print(f"{'='*50}")

        if method_name == "mc_dropout":
            print(f"  MC samples: {method['samples']}")
        else:
            print(f"  Relative input error: {method['relative_input_error']}")

        metrics = method["uncertainty_metrics"]
        if metrics:
            for key, value in metrics.items():
                if isinstance(value, (int, float)):
                    print(f"  {key:30s}: {value:.6f}")

        flood_analysis = method.get("flood_analysis")
        if flood_analysis:
            print("\n  Flood vs Normal uncertainty:")
            print(
                f"    Overflow unc (flood):  {flood_analysis['overflow_uncertainty_flood']:.6f}"
            )
            print(
                f"    Overflow unc (normal): {flood_analysis['overflow_uncertainty_normal']:.6f}"
            )
            print(
                f"    Depth unc (flood):     {flood_analysis['depth_uncertainty_flood']:.6f}"
            )
            print(
                f"    Depth unc (normal):    {flood_analysis['depth_uncertainty_normal']:.6f}"
            )
            print(
                f"    Flood events: {flood_analysis['flood_events_count']}, "
                f"Normal events: {flood_analysis['normal_events_count']}"
            )

---
## 10. Save Results to Google Drive

Copies all generated artifacts back to Google Drive for persistent storage.

In [ ]:
# ── 10.1 Copy artifacts to Google Drive ──
import shutil

DRIVE_RESULTS_DIR = os.path.join(DRIVE_DATA_DIR, "results")
DRIVE_CHECKPOINTS_DIR = os.path.join(DRIVE_DATA_DIR, "checkpoints")

# Copy results
src_results = Path("artifacts/results")
if src_results.exists():
    dst = Path(DRIVE_RESULTS_DIR)
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src_results, dst)
    print(f"Results copied to: {dst}")

# Copy checkpoints
src_checkpoints = Path("artifacts/checkpoints")
if src_checkpoints.exists():
    dst = Path(DRIVE_CHECKPOINTS_DIR)
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src_checkpoints, dst)
    print(f"Checkpoints copied to: {dst}")

# List what was saved
print("\nSaved artifacts on Google Drive:")
for root, dirs, files in os.walk(DRIVE_DATA_DIR):
    level = root.replace(DRIVE_DATA_DIR, "").count(os.sep)
    indent = " " * 2 * level
    basename = os.path.basename(root)
    print(f"{indent}{basename}/")
    subindent = " " * 2 * (level + 1)
    for file in files:
        filepath = os.path.join(root, file)
        size_mb = os.path.getsize(filepath) / 1e6
        print(f"{subindent}{file} ({size_mb:.1f} MB)")

In [ ]:
# ── 10.2 Create a downloadable archive of all results ──
import tarfile
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
results_archive = f"/content/bellinge_results_{timestamp}.tar.gz"

with tarfile.open(results_archive, "w:gz") as tar:
    tar.add("artifacts/results", arcname="results")
    tar.add("artifacts/checkpoints", arcname="checkpoints")
    if Path("artifacts/thresholds").exists():
        tar.add("artifacts/thresholds", arcname="thresholds")

archive_size_mb = os.path.getsize(results_archive) / 1e6
print(f"\n📦 Results archive: {results_archive} ({archive_size_mb:.1f} MB)")
print("   Download from Colab Files panel or find it on Drive.")

# Also copy to Drive
drive_archive = os.path.join(DRIVE_DATA_DIR, f"bellinge_results_{timestamp}.tar.gz")
shutil.copy2(results_archive, drive_archive)
print(f"   Copied to Drive: {drive_archive}")

---
## Summary

Pipeline complete. All artifacts are saved both locally in the Colab runtime and on Google Drive.

### Generated artifacts:

| Artifact | Path |
|----------|------|
| Single LNN checkpoint | `artifacts/checkpoints/lnn/best_model.pt` |
| Ensemble checkpoints | `artifacts/checkpoints/ensemble/lnn/seed_*/best_model.pt` |
| Threshold artifact | `artifacts/thresholds/lnn_val_threshold.json` |
| Evaluation metrics | `artifacts/results/release/` |
| Ensemble manifest | `artifacts/results/ensemble/lnn/` |
| Robustness results | `artifacts/results/robustness/lnn_robustness.json` |
| IG attributions | `artifacts/results/ig/lnn_integrated_gradients.json` |
| Uncertainty results | `artifacts/results/uncertainty/` |